โมดูล 4 · สรุปตัวเลขให้ถูก

เป้าหมาย: จบโมดูลนี้ Om จะเขียนประโยคสรุปเวลาส่งได้แบบที่ไม่หลอกคนอ่าน เช่น "มัธยฐาน 10.2 วัน (IQR a–b, n = …) และ 1 ใน 10 รอเกิน x วัน" และอธิบายได้ว่าทำไมเลือกตัวเลขพวกนี้

หัวข้อ	เรื่อง	Om รู้อะไรอยู่แล้ว
1. mean, median, mode	เห็นแล้วว่า mean 12.56 กับ median 10.22 ต่างกัน
2. range, percentile, IQR, P90	+ temp table และ view ที่ตกลงไว้ว่าจะเรียนตรงนี้
3. variance, SD	เรื่อง ddof จากโมดูล 3
4. five-number summary, boxplot	describe() ที่ใช้มาตลอด
5. skew, log	รู้แล้วว่าเบ้ขวา
6. outlier	ออเดอร์ 209 วัน
7. เลือกตัวสรุปให้เหมาะ	ประโยครายงาน

In [ ]:
import pandas as pd
import numpy as np
from olist_analysis import load, config, cleaning

cfg = config.ReportConfig
order = load.read_table('orders', cfg.raw_dir)
clean = cleaning.clean_orders(order)
review = load.read_table('order_reviews', cfg.raw_dir)
customer = load.read_table('customer', cfg.raw_dir)

in_scope = (
    clean['delivery_days'].notna()
    & (clean['order_purchase_timestamp'] >= cfg.start)
    & (clean['order_purchase_timestamp'] < cfg.end)
)

dv = clean[in_scope]
d= dv['delivery_days']
len(d)

4.1 ค่ากลางสามแบบ

In [ ]:
print(d.mean(), d.median(), d.round().mode().tolist())

d_trim = d[d <=100]
print(len(d) - len(d_trim), 'จำนวนค่าที่โดนตัดออก')
print(d_trim.mean(), d_trim.median())


	ก่อนตัด	หลังตัด 64 ออเดอร์	ขยับ
mean	12.539	12.451	0.088
median	10.210	10.206	0.004

ทั้งสองขยับน้อยในตัวเลขจริง เพราะ 64 ตัวเป็นแค่ราว 0.07% ของทั้งหมด แต่ถ้าดูอัตราส่วน mean ขยับมากกว่า median ราว 20 เท่า จากการตัดแค่ 64 ตัว

ผลนี้สำคัญมากตอนแบ่งกลุ่มย่อย สมมติรัฐหนึ่งมีออเดอร์แค่ 300 ตัว แล้วมี 3 ตัวที่ใช้เวลา 150 วัน mean ของรัฐนั้นจะพุ่งขึ้นหลายวัน แต่ median แทบไม่ขยับ ยิ่งกลุ่มเล็ก ค่าสุดขั้วยิ่งมีอิทธิพลต่อ mean มาก median จึงเรียกว่าตัวสรุปที่ ทน (robust)

4.2 Percentile, IQR และ P90

In [ ]:
q = d.quantile([0.25, 0.5, 0.75, 0.9])

print(q)
print('IQR = ', q[0.75]-q[0.25])
print('rage = ', d.max()-d.min())

หาว่ารีวิวการส่งช้า มีความเกี่ยวข้องกับ รีวิวแย่ๆ แค่ไหน

In [ ]:
last = (review.sort_values('review_answer_timestamp')
            .drop_duplicates('order_id', keep='last'))

r = dv.merge(last[['order_id', 'review_score']], on='order_id', how='inner')

p90 = r['delivery_days'].quantile(0.9)
r['slow_10pct'] = r['delivery_days'] > p90
r['one_star'] = r['review_score'] == 1

r.groupby('slow_10pct').agg(
    n=('order_id', 'size'),
    one_star_rate=('one_star', 'mean'),
    mean_score=('review_score', 'mean'),
)

4.3 Variance และ SD: การแกว่ง

In [ ]:
dvs = dv.merge(customer[['customer_id','customer_state']], on='customer_id')

g = dvs.groupby('customer_state')['delivery_days']

t = pd.DataFrame({
    "n": g.size(),
    "median": g.median(),
    "sd": g.std(),
    "IQR": g.quantile(0.75) - g.quantile(0.25)
})

t['cv'] = t['sd'] / g.mean()

t[t["n"] >= cfg.min_n].sort_values('sd').round(2)

t["sd_per_iqr"] = t["sd"] / t["IQR"]
t[t["n"] >= cfg.min_n].sort_values("sd_per_iqr", ascending=False).round(2).head(5)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Thonburi"      # ฟอนต์ไทยที่มีใน Mac

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(d[d <= 60], bins=60, color="#c9dcf3")
for x, label, color in [(d.median(), "median", "#eb6834"),
                        (d.mean(), "mean", "#2a78d6"),
                        (d.quantile(0.9), "P90", "#6250d6")]:
    ax.axvline(x, color=color, lw=2)
    ax.text(x + 0.5, ax.get_ylim()[1] * 0.9, f"{label} {x:.1f}", color="black")
ax.set_xlabel("เวลาส่ง (วัน)")
ax.set_ylabel("จำนวนออเดอร์")
plt.show()

4.4 Five-number summary และ boxplot

In [ ]:
top = dvs["customer_state"].value_counts().head(10).index
sub = dvs[dvs["customer_state"].isin(top)]
states = sub.groupby("customer_state")["delivery_days"].median().sort_values().index
data = [sub.loc[sub['customer_state'] == s, 'delivery_days'] for s in states]

fig, ax = plt.subplots(figsize=(10,5))
ax.boxplot(data, vert=False, showfliers=False, widths=0.6)
ax.invert_yaxis()
ax.set_yticks(range(1,len(states)+1),states)
ax.set_xlabel('เวลาส่ง(วัน))')
ax.set_title('เวลาส่งแยกรัฐ 10 รัฐที่มีออเดอร์มากที่สุด (ไม่แสดงจุด outlier)')

4.5 skew and log

In [ ]:
items = load.read_table("order_items", cfg.raw_dir)
v = items.groupby("order_id")["price"].sum()      # ยอดสินค้าต่อออเดอร์

print(f"skew: ปกติ {v.skew():.2f}   log {np.log10(v).skew():.2f}")
print(f"mean {v.mean():.1f}   median {v.median():.1f}   geometric mean {10 ** np.log10(v).mean():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(v[v <= 1000], bins=50)
axes[0].set_title("ยอดต่อออเดอร์ (สเกลปกติ)")
axes[1].hist(np.log10(v), bins=50)
axes[1].set_title("ยอดต่อออเดอร์ (log10)")
plt.show()

ภาพ log มีแท่งหยัก ขึ้น ๆ ลง ๆ ไม่เรียบเหมือนระฆังในภาพตัวอย่าง สาเหตุน่าจะเป็นเพราะร้านค้าชอบตั้งราคาเป็นตัวเลขกลม ๆ เช่น 49.90 หรือ 99.90 ราคาจึงกระจุกอยู่บางจุด แท่งที่ครอบจุดพวกนั้นจะสูงผิดปกติ นี่คือรายละเอียดที่ข้อมูลจำลองไม่มี (ถ้าอยากเช็ก ลอง items["price"].value_counts().head(10) จะเห็นราคาที่ซ้ำกันบ่อยที่สุด)

In [ ]:
items["price"].value_counts().head(10)

4.6 outlier 

In [ ]:
q1, q3 = d.quantile([0.25, 0.75])
fence_iqr = q3 + 1.5 * (q3 - q1)
fence_z = d.mean() + 3 * d.std()

print(f"รั้ว IQR {fence_iqr:.1f} วัน → {(d > fence_iqr).sum():,} ออเดอร์")
print(f"รั้ว z   {fence_z:.1f} วัน → {(d > fence_z).sum():,} ออเดอร์")

top6 = d.nlargest(6)
d_cut = d.drop(top6.index)
fence_z_cut = d_cut.mean() + 3 * d_cut.std()
print(f"ตัด 6 ตัวที่นานที่สุด รั้ว z {fence_z_cut:.1f} วัน → {(d_cut > fence_z_cut).sum():,} ออเดอร์")